# 05 - D-MAD Evaluation

This notebook evaluates RRPR perturbations using Differential Morphing Attack Detection (D-MAD).

Face Recognition Systems:

- AdaFace
- ArcFace
- MagFace
- ElasticFace
- EdgeFace

Datasets:

- FERET
- FRGC

Inputs:

- Embeddings_Diff/

Outputs:

- DMAD_Results/
- Table 2

Metrics:

- DEER
- BPCER@APCER=5%

Expected Runtime:

Demo Mode:
- 50 minutes - 1 hour

Full Dataset:
- 3-4 hours

GPU Required:

- No

In [3]:
try:
    from google.colab import drive

    drive.mount("/content/drive")

    print("Google Drive mounted.")

except ImportError:

    print("Running outside Colab. Drive mount skipped.")

Mounted at /content/drive


In [4]:
from pathlib import Path
import os
import glob
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_curve

warnings.filterwarnings("ignore")

## Configuration

This section defines:

- Delta embedding locations
- FRS models
- Datasets
- Perturbation types
- Morph generators

In [5]:
PROJECT_ROOT = Path("/content/drive/MyDrive/Free-Cloud/ICPR_Rep")
DMAD_ROOT    = PROJECT_ROOT / "Embeddings_Diff"
RESULTS_ROOT = PROJECT_ROOT / "DMAD_Results"

RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print("Delta Root:")
print(DMAD_ROOT)

print("\nResults Root:")
print(RESULTS_ROOT)

Delta Root:
/content/drive/MyDrive/Free-Cloud/ICPR_Rep/Embeddings_Diff

Results Root:
/content/drive/MyDrive/Free-Cloud/ICPR_Rep/DMAD_Results


In [4]:
MODELS = ["adaface", "arcface", "magface", "elasticface", "edgeface"]

DATASETS = ["FERET", "FRGC"]

MORPH_TYPES = ["greedy", "mipgan2", "ubo"]

PERTURBATIONS = [
    "original",
    "Weighted_Ensemble_Template_PGD",
    "Weighted_Ensemble_Template_DCT_HF",
    "Weighted_Ensemble_Template_DWT_HF",
    "Weighted_Ensemble_Template_BPDA_EOT"
]

TARGET_APCERS = [0.05]

## Delta Embeddings

D-MAD is performed using delta embeddings generated in:

03_embedding_and_delta_embeddings.ipynb

Inputs:

- Genuine delta embeddings
- Morph delta embeddings

In [5]:
# Utility
def load_npy_files_from_folder(folder):
    folder = Path(folder)
    if not folder.exists():
        return []
    return sorted(folder.rglob("*.npy"))


def load_features_from_paths(paths):
    X = []
    y = []
    f = []

    for path in paths:
        path = Path(path)

        if path.is_dir():
            files = load_npy_files_from_folder(path)
        else:
            files = [path]

        for file in files:
            try:
                feat = np.load(file).reshape(-1)
            except:
                continue

            file_str = str(file)

            if "/aligned/" in file_str:
                label = 0
            elif "/morph/" in file_str:
                label = 1
            else:
                continue

            X.append(feat)
            y.append(label)
            f.append(str(file))

    if len(X) == 0:
        return (np.empty((0,)), np.array([]), [])

    return (np.stack(X), np.array(y), f)

In [6]:
# Metrics

def calculate_eer_from_scores(bona_scores, attack_scores):
    if len(bona_scores) + len(attack_scores) < 2:
        return np.nan

    y = np.concatenate([
        np.zeros(len(bona_scores)),
        np.ones(len(attack_scores))
    ])

    scores = np.concatenate([bona_scores, attack_scores])

    fpr, tpr, _ = roc_curve(y, scores, pos_label=1)
    fnr = 1 - tpr

    idx = np.nanargmin(np.abs(fpr - fnr))
    eer = (fpr[idx] + fnr[idx]) / 2.0

    return eer * 100.0


def threshold_for_apcer(attack_scores, target_apcer):
    if len(attack_scores) == 0:
        return None

    percentile = np.clip(target_apcer * 100, 0, 100)
    return float(np.percentile(attack_scores, percentile))


def compute_bpcer_at_threshold(bona_scores, threshold):
    if threshold is None or len(bona_scores) == 0:
        return np.nan

    return float((np.array(bona_scores) >= threshold).mean() * 100.0)

## D-MAD Classifier

A Radial Basis Function (RBF) Support Vector Machine is used.

Configuration:

- Kernel = RBF
- Class Weight = Balanced

The balanced setting compensates for unequal numbers of:

- Bona fide samples
- Morph samples

In [8]:
# Train SVM

def train_dmad_svm(train_genuine_paths, train_morph_paths):
    X_genuine, y_genuine, _ = load_features_from_paths(train_genuine_paths)
    X_morph,   y_morph,   _ = load_features_from_paths(train_morph_paths)

    X_train = np.vstack([X_genuine, X_morph])
    y_train = np.concatenate([y_genuine, y_morph])

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)

    svm = SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42)
    svm.fit(X_train_scaled, y_train)

    return (svm, scaler, len(X_genuine), len(X_morph))

## Training Protocol

Training uses:

original/aligned/train

and

original/morph/

including:

- Greedy
- MIPGAN-II
- UBO

Only original training data are used for classifier fitting.

In [9]:
def evaluate_dmad(svm, scaler, bona_paths, morph_paths):
    X_bona,  _, _ = load_features_from_paths(bona_paths)
    X_morph, _, _ = load_features_from_paths(morph_paths)

    X_bona  = scaler.transform(X_bona)
    X_morph = scaler.transform(X_morph)

    bona_scores  = svm.predict_proba(X_bona)[:, 1]
    morph_scores = svm.predict_proba(X_morph)[:, 1]

    eer = calculate_eer_from_scores(bona_scores, morph_scores)
    results = {"DEER": eer}

    for target_apcer in TARGET_APCERS:
        threshold = threshold_for_apcer(morph_scores, target_apcer)
        bpcer = compute_bpcer_at_threshold(bona_scores, threshold)
        results[f"BPCER@{int(target_apcer * 100)}"] = bpcer

    return results

In [10]:
def get_training_paths(model, dataset):
    root = DMAD_ROOT / model / dataset / "original"

    genuine_train = [root / "aligned" / "train"]

    morph_train = [root / "morph" / morph_type / "train" for morph_type in MORPH_TYPES]

    return (genuine_train, morph_train)

## Testing Protocol

Evaluation is performed independently on:

- Original
- PGD
- DCT-HF
- DWT-HF
- BPDA-EOT

No training data are used during testing.

In [11]:
def get_test_paths(model, dataset, perturbation, morph_type):
    root = DMAD_ROOT / model / dataset / perturbation

    bona_test  = [root / "aligned" / "test"]
    morph_test = [root / "morph" / morph_type / "test"]

    return (bona_test, morph_test)

## Metrics

Reported metrics:

### DEER

Differential Equal Error Rate.

### BPCER@APCER=5%

Bona Fide Presentation Classification Error Rate
at

APCER = 5%.

## Evaluation

The following combinations are evaluated:

Datasets:
- FERET
- FRGC

FRS:
- AdaFace
- ArcFace
- MagFace
- ElasticFace
- EdgeFace

Perturbations:
- Original
- PGD
- DCT-HF
- DWT-HF
- BPDA-EOT

Morph Generators:
- Greedy
- MIPGAN-II
- UBO

In [12]:
results = []

for model in MODELS:
    print("\n" + "=" * 80)
    print(model)
    print("=" * 80)

    for dataset in DATASETS:
        print(f"\nTraining: {model} | {dataset}")

        train_genuine_paths, train_morph_paths = get_training_paths(model, dataset)
        svm, scaler, n_genuine, n_morph = train_dmad_svm(train_genuine_paths, train_morph_paths)

        print(f"Genuine: {n_genuine}")
        print(f"Morph: {n_morph}")

        for perturbation in PERTURBATIONS:
            for morph_type in MORPH_TYPES:
                bona_test, morph_test = get_test_paths(model, dataset, perturbation, morph_type)
                metrics = evaluate_dmad(svm, scaler, bona_test, morph_test)

                results.append({
                    "model": model,
                    "dataset": dataset,
                    "perturbation": perturbation,
                    "morph_type": morph_type,
                    **metrics
                })

                print(dataset, perturbation, morph_type, metrics["DEER"])


adaface

Training: adaface | FERET
Genuine: 38
Morph: 456
FERET original greedy 0.0
FERET original mipgan2 0.0
FERET original ubo 0.0
FERET Weighted_Ensemble_Template_PGD greedy 61.25000000000001
FERET Weighted_Ensemble_Template_PGD mipgan2 19.02777777777778
FERET Weighted_Ensemble_Template_PGD ubo 60.555555555555564
FERET Weighted_Ensemble_Template_DCT_HF greedy 42.22222222222222
FERET Weighted_Ensemble_Template_DCT_HF mipgan2 0.0
FERET Weighted_Ensemble_Template_DCT_HF ubo 38.75
FERET Weighted_Ensemble_Template_DWT_HF greedy 42.22222222222222
FERET Weighted_Ensemble_Template_DWT_HF mipgan2 21.11111111111111
FERET Weighted_Ensemble_Template_DWT_HF ubo 37.361111111111114
FERET Weighted_Ensemble_Template_BPDA_EOT greedy 57.08333333333333
FERET Weighted_Ensemble_Template_BPDA_EOT mipgan2 19.02777777777778
FERET Weighted_Ensemble_Template_BPDA_EOT ubo 59.16666666666666

Training: adaface | FRGC
Genuine: 298
Morph: 1320
FRGC original greedy 3.035714285714285
FRGC original mipgan2 0.416666

In [13]:
results_df = pd.DataFrame(results)

raw_csv = RESULTS_ROOT / "dmad_raw_results.csv"
results_df.to_csv(raw_csv, index=False)

print(raw_csv)
display(results_df.head())

/content/drive/MyDrive/Free-Cloud/ICPR_Rep/DMAD_Results/dmad_raw_results.csv


,model,dataset,perturbation,morph_type,DEER,BPCER@5
0,adaface,FERET,original,greedy,0.000000,0.0
1,adaface,FERET,original,mipgan2,0.000000,0.0
2,adaface,FERET,original,ubo,0.000000,0.0
3,adaface,FERET,Weighted_Ensemble_Template_PGD,greedy,61.250000,100.0
4,adaface,FERET,Weighted_Ensemble_Template_PGD,mipgan2,19.027778,20.0


In [14]:
from scipy.stats import t

def mean_ci_95(values):
    values = np.asarray(values, dtype=float)
    values = values[~np.isnan(values)]
    n = len(values)

    if n == 0:
        return np.nan, np.nan

    mean = np.mean(values)

    if n == 1:
        return mean, 0.0

    sem = np.std(values, ddof=1) / np.sqrt(n)
    ci = t.ppf(0.975, n - 1) * sem

    return mean, ci

## Aggregation

Results are aggregated across:

- Greedy
- MIPGAN-II
- UBO

and reported as:

mean ± 95% confidence interval.

In [16]:
aggregated_rows = []

group_cols = ["model", "dataset", "perturbation"]

for keys, group_df in results_df.groupby(group_cols):
    model, dataset, perturbation = keys

    row = {"model": model, "dataset": dataset, "perturbation": perturbation}

    deer_mean, deer_ci = mean_ci_95(group_df["DEER"])
    row["DEER_mean"] = deer_mean
    row["DEER_ci"]   = deer_ci

    for metric in ["BPCER@5"]:
        mean_val, ci_val = mean_ci_95(group_df[metric])
        row[f"{metric}_mean"] = mean_val
        row[f"{metric}_ci"]   = ci_val

    aggregated_rows.append(row)

aggregated_df = pd.DataFrame(aggregated_rows)
aggregated_df = aggregated_df.sort_values(["model", "dataset", "perturbation"]).reset_index(drop=True)

display(aggregated_df.head())

,model,dataset,perturbation,DEER_mean,DEER_ci,BPCER@5_mean,BPCER@5_ci
0,adaface,FERET,Weighted_Ensemble_Template_BPDA_EOT,45.092593,56.133598,73.333333,28.684352
1,adaface,FERET,Weighted_Ensemble_Template_DCT_HF,26.990741,58.225832,66.666667,143.421758
2,adaface,FERET,Weighted_Ensemble_Template_DWT_HF,33.564815,27.463898,80.000000,86.053055
3,adaface,FERET,Weighted_Ensemble_Template_PGD,46.944444,60.064055,73.333333,114.737406
4,adaface,FERET,original,0.000000,0.000000,0.000000,0.000000


In [17]:
agg_csv = RESULTS_ROOT / "dmad_mean_ci_results.csv"
aggregated_df.to_csv(agg_csv, index=False)
print(agg_csv)

/content/drive/MyDrive/Free-Cloud/ICPR_Rep/DMAD_Results/dmad_mean_ci_results.csv


In [18]:
def format_mean_ci(mean, ci, decimals=2):
    if np.isnan(mean):
        return "-"
    return f"{mean:.{decimals}f} ± {ci:.{decimals}f}"

In [19]:
# BPCER@5

dataset_name = "FERET"
paper_rows = []

for model in MODELS:
    subset = aggregated_df[
        (aggregated_df["dataset"] == dataset_name) &
        (aggregated_df["model"] == model)
    ]

    row = {"Model": model}

    for perturbation in PERTURBATIONS:
        p_df = subset[subset["perturbation"] == perturbation]

        if len(p_df) == 0:
            row[perturbation] = "-"
            continue

        row[perturbation] = format_mean_ci(p_df.iloc[0]["BPCER@5_mean"], p_df.iloc[0]["BPCER@5_ci"])

    paper_rows.append(row)

bpcer_table = pd.DataFrame(paper_rows)
display(bpcer_table)

,Model,original,Weighted_Ensemble_Template_PGD,Weighted_Ensemble_Template_DCT_HF,Weighted_Ensemble_Template_DWT_HF,Weighted_Ensemble_Template_BPDA_EOT
0,adaface,0.00 ± 0.00,73.33 ± 114.74,66.67 ± 143.42,80.00 ± 86.05,73.33 ± 28.68
1,arcface,0.00 ± 0.00,73.33 ± 114.74,66.67 ± 143.42,80.00 ± 86.05,73.33 ± 75.89
2,magface,6.67 ± 28.68,73.33 ± 114.74,73.33 ± 114.74,66.67 ± 57.37,80.00 ± 49.68
3,elasticface,0.00 ± 0.00,80.00 ± 86.05,53.33 ± 114.74,73.33 ± 28.68,73.33 ± 28.68
4,edgeface,0.00 ± 0.00,66.67 ± 103.42,66.67 ± 143.42,46.67 ± 57.37,73.33 ± 28.68


In [ ]:
## Final Table

## Paper Table Generation

This section generates the final D-MAD table reported in the paper.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

In [6]:
RESULTS_ROOT = PROJECT_ROOT / "DMAD_Results"

aggregated_df = pd.read_csv(RESULTS_ROOT / "dmad_mean_ci_results.csv")

print(aggregated_df.shape)
display(aggregated_df.head())

(50, 7)


,model,dataset,perturbation,DEER_mean,DEER_ci,BPCER@5_mean,BPCER@5_ci
0,adaface,FERET,Weighted_Ensemble_Template_BPDA_EOT,45.092593,56.133598,73.333333,28.684352
1,adaface,FERET,Weighted_Ensemble_Template_DCT_HF,26.990741,58.225832,66.666667,143.421758
2,adaface,FERET,Weighted_Ensemble_Template_DWT_HF,33.564815,27.463898,80.000000,86.053055
3,adaface,FERET,Weighted_Ensemble_Template_PGD,46.944444,60.064055,73.333333,114.737406
4,adaface,FERET,original,0.000000,0.000000,0.000000,0.000000


In [7]:
PERT_ORDER = [
    "original",
    "Weighted_Ensemble_Template_DCT_HF",
    "Weighted_Ensemble_Template_DWT_HF",
    "Weighted_Ensemble_Template_BPDA_EOT",
    "Weighted_Ensemble_Template_PGD"
]

MODEL_ORDER = ["adaface", "magface", "arcface", "elasticface", "edgeface"]

PERT_NAME_MAP = {
    "original":                            "Original",
    "Weighted_Ensemble_Template_DCT_HF":   "DCT-HF",
    "Weighted_Ensemble_Template_DWT_HF":   "DWT-HF",
    "Weighted_Ensemble_Template_BPDA_EOT": "BPDA-EOT",
    "Weighted_Ensemble_Template_PGD":      "PGD"
}

MODEL_NAME_MAP = {
    "adaface":     "AdaFace",
    "magface":     "MagFace",
    "arcface":     "ArcFace",
    "elasticface": "ElasticFace",
    "edgeface":    "EdgeFace"
}

In [8]:
def fmt(mean, ci):
    if pd.isna(mean):
        return "-"
    return f"{mean:.4f} ± {ci:.4f}"

In [9]:
# Paper Table

table_rows = []

for perturbation in PERT_ORDER:
    for model in MODEL_ORDER:
        feret_row = aggregated_df[
            (aggregated_df["dataset"] == "FERET") &
            (aggregated_df["perturbation"] == perturbation) &
            (aggregated_df["model"] == model)
        ]

        frgc_row = aggregated_df[
            (aggregated_df["dataset"] == "FRGC") &
            (aggregated_df["perturbation"] == perturbation) &
            (aggregated_df["model"] == model)
        ]

        if len(feret_row) == 0 or len(frgc_row) == 0:
            continue

        table_rows.append({
            "Perturbation": PERT_NAME_MAP[perturbation],
            "FRS":          MODEL_NAME_MAP[model],
            "FERET":        fmt(feret_row.iloc[0]["BPCER@5_mean"], feret_row.iloc[0]["BPCER@5_ci"]),
            "FRGC":         fmt(frgc_row.iloc[0]["BPCER@5_mean"],  frgc_row.iloc[0]["BPCER@5_ci"])
        })

paper_table = pd.DataFrame(table_rows)
display(paper_table)

,Perturbation,FRS,FERET,FRGC
0,Original,AdaFace,0.0000 ± 0.0000,2.5000 ± 6.2103
1,Original,MagFace,6.6667 ± 28.6844,0.0000 ± 0.0000
2,Original,ArcFace,0.0000 ± 0.0000,2.2222 ± 3.1622
3,Original,ElasticFace,0.0000 ± 0.0000,0.5556 ± 1.1952
4,Original,EdgeFace,0.0000 ± 0.0000,3.8889 ± 1.1952
5,DCT-HF,AdaFace,66.6667 ± 143.4218,23.0556 ± 47.9862
6,DCT-HF,MagFace,73.3333 ± 114.7374,15.8333 ± 33.8892
7,DCT-HF,ArcFace,66.6667 ± 143.4218,10.0000 ± 21.8100
8,DCT-HF,ElasticFace,53.3333 ± 114.7374,11.3889 ± 24.5231
9,DCT-HF,EdgeFace,66.6667 ± 143.4218,4.7222 ± 8.3663


## Generated Outputs
``` text
DMAD_Results/

├── dmad_raw_results.csv
├── dmad_mean_ci_results.csv
└── table2_dmad_paper.csv
```
These files reproduce the D-MAD results reported in the paper.